# Lesson 9: AI Incident Management
## Exercise: Threshold-Based Incident Detection for TransitAI

**Quick Reference — Incident Management Concepts:**

| Concept | Description |
|---------|-------------|
| **Severity Levels** | S1 (Critical, <15min) → S2 (High, <1hr) → S3 (Medium, <4hr) → S4 (Low, <24hr) |
| **Detection Methods** | Threshold-based, anomaly detection, user reports, automated monitoring |
| **Response Phases** | Detect → Triage → Classify → Contain → Investigate → Resolve → Review |
| **Key Metrics** | Model accuracy, response latency, bias score, error rate, uptime |
| **Escalation Triggers** | Safety risk, data breach, widespread impact, regulatory implication |

**Scenario:** TransitAI operates autonomous route optimization across 15 metro areas serving 2M+ daily passengers. You'll implement incident detection logic for their monitoring system.

**Your Task:** This is the complete solution notebook with all sections implemented. that require incident management thinking (~10-15 min).

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("TransitAI Incident Detection System — Libraries loaded")

## Monitoring Data (Pre-loaded)
100 hours of TransitAI system monitoring data with 7 injected anomalies simulating real incidents.

In [ ]:
# Generate 100 hours of monitoring data
n_samples = 100
timestamps = [datetime.now() - timedelta(hours=100-i) for i in range(n_samples)]

monitoring_data = pd.DataFrame({
    'timestamp': timestamps,
    'model_accuracy': np.random.normal(94, 2, n_samples).clip(70, 100),
    'response_latency_ms': np.random.normal(120, 15, n_samples).clip(50, 500),
    'bias_score': np.random.normal(0.05, 0.02, n_samples).clip(0, 1),
    'error_rate': np.random.normal(0.02, 0.005, n_samples).clip(0, 1),
    'uptime_pct': np.random.normal(99.9, 0.1, n_samples).clip(95, 100),
    'content_safety_score': np.random.normal(0.98, 0.01, n_samples).clip(0, 1)
})

# Inject realistic anomalies
anomalies = [
    {'name': 'Accuracy Drop (Performance)', 'start': 10, 'end': 15, 'metric': 'model_accuracy', 'value': 78, 'severity': 'S2'},
    {'name': 'Latency Spike (Load)', 'start': 22, 'end': 27, 'metric': 'response_latency_ms', 'value': 280, 'severity': 'S2'},
    {'name': 'Bias Score Increase', 'start': 35, 'end': 40, 'metric': 'bias_score', 'value': 0.25, 'severity': 'S2'},
    {'name': 'Error Rate Spike', 'start': 50, 'end': 55, 'metric': 'error_rate', 'value': 0.15, 'severity': 'S3'},
    {'name': 'Uptime Drop (Outage)', 'start': 65, 'end': 68, 'metric': 'uptime_pct', 'value': 96.5, 'severity': 'S1'},
    {'name': 'Accuracy + Latency Combined', 'start': 80, 'end': 85, 'metric': 'model_accuracy', 'value': 82, 'severity': 'S2'},
    {'name': 'Latency Spike #2', 'start': 80, 'end': 85, 'metric': 'response_latency_ms', 'value': 250, 'severity': 'S2'},
    {'name': 'Content Safety Drop (Hallucination)', 'start': 42, 'end': 47, 'metric': 'content_safety_score', 'value': 0.88, 'severity': 'S2'},
]

for anomaly in anomalies:
    monitoring_data.loc[anomaly['start']:anomaly['end'], anomaly['metric']] = anomaly['value'] + np.random.normal(0, 1, anomaly['end'] - anomaly['start'] + 1)

print(f"Monitoring data: {len(monitoring_data)} hours, {len(monitoring_data.columns)-1} metrics")
print(f"Injected anomalies: {len(anomalies)}")
monitoring_data.head(3)

## Detection Thresholds (Pre-defined)

In [ ]:
thresholds = {
    'model_accuracy': {'warning': 90, 'critical': 85, 'direction': 'below', 'unit': '%'},
    'response_latency_ms': {'warning': 180, 'critical': 250, 'direction': 'above', 'unit': 'ms'},
    'bias_score': {'warning': 0.10, 'critical': 0.20, 'direction': 'above', 'unit': 'score'},
    'error_rate': {'warning': 0.05, 'critical': 0.10, 'direction': 'above', 'unit': 'rate'},
    'uptime_pct': {'warning': 99.5, 'critical': 99.0, 'direction': 'below', 'unit': '%'},
    'content_safety_score': {'warning': 0.95, 'critical': 0.90, 'direction': 'below', 'unit': 'score'},
}

print("Detection Thresholds:")
for metric, t in thresholds.items():
    print(f"  {metric}: warning={t['warning']}{t['unit']}, critical={t['critical']}{t['unit']} ({t['direction']})")

## Part 3: Incident Detection Logic (Solution)

Complete the `detect_incidents()` function. For each data point:
1. Compare the metric value against warning and critical thresholds
2. Check the `direction` field — 'below' means alert when value < threshold, 'above' means alert when value > threshold
3. Classify as 'warning' or 'critical' based on which threshold is breached
4. Return a list of incident dictionaries

**Governance thinking:** Why do we use two threshold levels (warning vs critical)? How does this map to the severity classification in your Excel playbook?

In [ ]:
def detect_incidents(data, thresholds):
    """
    Detect incidents by comparing metrics against thresholds.
    Includes S1 (critical), S2 (compound breaches), S3 (warning), and S4 (approaching) detection.
    
    Args:
        data: DataFrame with monitoring metrics
        thresholds: Dict with warning/critical thresholds per metric
    
    Returns:
        List of incident dicts with: timestamp, metric, value, threshold_type, threshold_value, severity
    """
    incidents = []
    
    # Track breaching metrics per timestamp for compound incident detection
    metrics_by_timestamp = {}
    
    for idx, row in data.iterrows():
        timestamp = row['timestamp']
        metrics_by_timestamp[timestamp] = {'breaching_metrics': [], 'approaching_metrics': []}
        
        for metric, thresh in thresholds.items():
            value = row[metric]
            direction = thresh['direction']
            
            # Check critical threshold first (higher priority)
            if direction == 'below':
                if value < thresh['critical']:
                    incidents.append({
                        'timestamp': timestamp, 'metric': metric, 'value': round(value, 3),
                        'threshold_type': 'critical', 'threshold_value': thresh['critical'], 'severity': 'S1'
                    })
                    metrics_by_timestamp[timestamp]['breaching_metrics'].append((metric, 'critical'))
                elif value < thresh['warning']:
                    incidents.append({
                        'timestamp': timestamp, 'metric': metric, 'value': round(value, 3),
                        'threshold_type': 'warning', 'threshold_value': thresh['warning'], 'severity': 'S3'
                    })
                    metrics_by_timestamp[timestamp]['breaching_metrics'].append((metric, 'warning'))
                # S4 detection: within 10% of warning threshold
                elif value < thresh['warning'] * 1.10:  # 10% above warning for 'below' direction
                    metrics_by_timestamp[timestamp]['approaching_metrics'].append(metric)
            else:  # direction == 'above'
                if value > thresh['critical']:
                    incidents.append({
                        'timestamp': timestamp, 'metric': metric, 'value': round(value, 3),
                        'threshold_type': 'critical', 'threshold_value': thresh['critical'], 'severity': 'S1'
                    })
                    metrics_by_timestamp[timestamp]['breaching_metrics'].append((metric, 'critical'))
                elif value > thresh['warning']:
                    incidents.append({
                        'timestamp': timestamp, 'metric': metric, 'value': round(value, 3),
                        'threshold_type': 'warning', 'threshold_value': thresh['warning'], 'severity': 'S3'
                    })
                    metrics_by_timestamp[timestamp]['breaching_metrics'].append((metric, 'warning'))
                # S4 detection: within 10% of warning threshold (approaching from below)
                elif value > thresh['warning'] * 0.90:  # 10% below warning for 'above' direction
                    metrics_by_timestamp[timestamp]['approaching_metrics'].append(metric)
    
    # S2 detection: Compound metrics — 2+ warning-level breaches simultaneously = S2
    # Also check for critical+warning combinations indicating systemic issues
    compound_incidents = []
    for idx, row in data.iterrows():
        timestamp = row['timestamp']
        if timestamp in metrics_by_timestamp:
            breaching = metrics_by_timestamp[timestamp]['breaching_metrics']
            warning_breaches = [m for m, level in breaching if level == 'warning']
            critical_breaches = [m for m, level in breaching if level == 'critical']
            
            # S2: 2+ warning breaches (systemic degradation)
            if len(warning_breaches) >= 2:
                # Create a compound incident entry (don't duplicate individual incidents)
                compound_incidents.append({
                    'timestamp': timestamp,
                    'metric': 'COMPOUND: ' + ' + '.join(sorted(set(warning_breaches))),
                    'value': None,
                    'threshold_type': 'compound',
                    'threshold_value': None,
                    'severity': 'S2'
                })
            
            # S2: Critical + Warning combo (systemic failure)
            if len(critical_breaches) >= 1 and len(warning_breaches) >= 1:
                compound_incidents.append({
                    'timestamp': timestamp,
                    'metric': 'COMPOUND: ' + ' + '.join(sorted(set(critical_breaches + warning_breaches))),
                    'value': None,
                    'threshold_type': 'compound',
                    'threshold_value': None,
                    'severity': 'S2'
                })
    
    # S4 detection: metrics approaching thresholds (simple version - track over time)
    # For this implementation, detect when approaching_metrics exist
    approaching_count_by_timestamp = {}
    for idx in range(len(data)):
        row = data.iloc[idx]
        timestamp = row['timestamp']
        approaching_count = len(metrics_by_timestamp.get(timestamp, {}).get('approaching_metrics', []))
        
        if approaching_count >= 1:
            if timestamp not in approaching_count_by_timestamp:
                approaching_count_by_timestamp[timestamp] = 0
            approaching_count_by_timestamp[timestamp] += 1
    
    # S4: Add advisory for approaching thresholds
    for idx in range(len(data) - 2):
        row = data.iloc[idx]
        timestamp = row['timestamp']
        next_ts = data.iloc[idx + 1]['timestamp']
        next_next_ts = data.iloc[idx + 2]['timestamp']
        
        if timestamp in approaching_count_by_timestamp and next_ts in approaching_count_by_timestamp:
            # If metrics are approaching for consecutive hours, flag as S4 advisory
            for metric, thresh in thresholds.items():
                value = row[metric]
                direction = thresh['direction']
                next_val = data.iloc[idx + 1][metric]
                
                # Simple S4 detection: value is between 90-110% of warning threshold
                if direction == 'below':
                    if thresh['warning'] * 0.90 < value < thresh['warning'] * 1.10:
                        next_next_val = data.iloc[idx + 2][metric]
                        if thresh['warning'] * 0.90 < next_val < thresh['warning'] * 1.10 and \
                           thresh['warning'] * 0.90 < next_next_val < thresh['warning'] * 1.10:
                            incidents.append({
                                'timestamp': timestamp, 'metric': metric, 'value': round(value, 3),
                                'threshold_type': 'approaching', 'threshold_value': thresh['warning'], 'severity': 'S4'
                            })
                            break  # Only add once per timestamp
                else:  # above
                    if thresh['warning'] * 0.90 < value < thresh['warning'] * 1.10:
                        next_next_val = data.iloc[idx + 2][metric]
                        if thresh['warning'] * 0.90 < next_val < thresh['warning'] * 1.10 and \
                           thresh['warning'] * 0.90 < next_next_val < thresh['warning'] * 1.10:
                            incidents.append({
                                'timestamp': timestamp, 'metric': metric, 'value': round(value, 3),
                                'threshold_type': 'approaching', 'threshold_value': thresh['warning'], 'severity': 'S4'
                            })
                            break  # Only add once per timestamp
    
    # Combine compound incidents with regular incidents
    incidents.extend(compound_incidents)
    
    return incidents

# Run detection
detected = detect_incidents(monitoring_data, thresholds)
print(f"Detected {len(detected)} incidents")

## Incident Report Generation (Pre-provided)

In [ ]:
if detected:
    incident_df = pd.DataFrame(detected)
    incident_df = incident_df.sort_values('timestamp')
    
    print("=" * 70)
    print("TRANSITAI INCIDENT REPORT")
    print("=" * 70)
    print(f"\nTotal incidents detected: {len(incident_df)}")
    print(f"\nBy severity:")
    # Dynamically iterate through all detected severities
    for sev in sorted(incident_df['severity'].unique()):
        count = len(incident_df[incident_df['severity'] == sev])
        sev_map = {'S1': 'Critical', 'S2': 'High', 'S3': 'Medium', 'S4': 'Low'}
        print(f"  {sev} ({sev_map.get(sev, 'Unknown')}): {count}")
    print(f"\nBy metric:")
    for metric in incident_df['metric'].unique():
        count = len(incident_df[incident_df['metric'] == metric])
        print(f"  {metric}: {count}")
    
    incident_df.head(10)
else:
    print("No incidents detected — check your detection logic!")
    incident_df = pd.DataFrame()

## Part 5: Escalation Decision Logic (Solution)

Complete the `determine_escalation()` function that decides WHO should be notified based on:
- The incident severity (S1 = critical, S2 = high, S3 = medium, S4 = low)
- The metric type (some metrics like uptime are more operationally critical)

**Governance thinking:** In a safety-critical transit system, who needs to know about different types of incidents? Consider regulatory reporting requirements (NTSB/FTA for transit).

In [ ]:
def determine_escalation(severity, metric):
    """
    Determine escalation path based on severity and metric type.
    
    Args:
        severity: 'S1', 'S2', 'S3', or 'S4'
        metric: Name of the affected metric
    
    Returns:
        Dict with: notify_list, response_sla, requires_war_room (bool), regulatory_report (bool)
    """
    escalation = {
        'S1': {
            'notify_list': ['CEO/Board', 'CTO', 'VP Engineering', 'Legal', 'Chief Security Officer', 'NTSB/FTA'],
            'response_sla': '< 15 minutes',
            'requires_war_room': True,
            'regulatory_report': True
        },
        'S2': {
            'notify_list': ['VP Engineering', 'VP Product', 'Chief Compliance Officer', 'Engineering Manager'],
            'response_sla': '< 1 hour',
            'requires_war_room': False,
            'regulatory_report': False
        },
        'S3': {
            'notify_list': ['Engineering Manager', 'Team Lead'],
            'response_sla': '< 4 hours',
            'requires_war_room': False,
            'regulatory_report': False
        },
        'S4': {
            'notify_list': ['Team Lead', 'Development Team'],
            'response_sla': '< 24 hours',
            'requires_war_room': False,
            'regulatory_report': False
        }
    }
    
    base = escalation.get(severity, escalation['S3'])
    result = base.copy()
    result['notify_list'] = list(result['notify_list'])
    
    # Metric-specific escalation adjustments
    if metric == 'bias_score' and severity != 'S1':
        if 'Ethics Review Board' not in result['notify_list']:
            result['notify_list'].append('Ethics Review Board')
        result['regulatory_report'] = True  # Bias in transit requires regulatory awareness
    
    if 'content_safety' in metric.lower():
        if 'Ethics Review Board' not in result['notify_list']:
            result['notify_list'].append('Ethics Review Board')
        if 'Content Safety Team' not in result['notify_list']:
            result['notify_list'].append('Content Safety Team')
        if severity in ['S1', 'S2']:
            result['regulatory_report'] = True
    
    if metric == 'uptime_pct':
        result['notify_list'].append('Operations Center')
        if severity == 'S1':
            result['notify_list'].append('Emergency Response Team')
    
    return result

# Apply to detected incidents
if len(incident_df) > 0:
    escalations = incident_df.apply(
        lambda row: determine_escalation(row['severity'], row['metric']), axis=1
    )
    escalation_df = pd.DataFrame(escalations.tolist())
    result_df = pd.concat([incident_df.reset_index(drop=True), escalation_df], axis=1)
    print(f"Escalation decisions for {len(result_df)} incidents:")
    result_df[['metric', 'severity', 'notify_list', 'response_sla', 'requires_war_room', 'regulatory_report']].head(10)

## Part 6: Response Recommendations (Solution)

Complete the `recommend_response()` function that suggests specific response actions based on the incident type (metric) and severity.

**Governance thinking:** What's the difference between containing an incident and resolving it? Why is the distinction important for a safety-critical transit system?

In [ ]:
def recommend_response(metric, severity):
    """
    Recommend response actions based on incident type and severity.
    
    Args:
        metric: The affected metric name
        severity: 'S1', 'S2', 'S3', or 'S4'
    
    Returns:
        Dict with: containment_actions (list), resolution_steps (list), post_incident (list)
    """
    responses = {
        'model_accuracy': {
            'containment_actions': ['Enable fallback rule-based routing', 'Scale compute resources', 'Alert operations team'],
            'resolution_steps': ['Profile model inference bottlenecks', 'Optimize for peak load', 'Implement request caching'],
            'post_incident': ['Add load testing to CI/CD', 'Establish performance SLAs', 'Implement auto-scaling']
        },
        'response_latency_ms': {
            'containment_actions': ['Enable CDN caching', 'Scale infrastructure', 'Activate circuit breakers'],
            'resolution_steps': ['Optimize database queries', 'Add compute capacity', 'Review API gateway config'],
            'post_incident': ['Implement latency budgets', 'Add performance regression tests', 'Set up auto-scaling policies']
        },
        'bias_score': {
            'containment_actions': ['Flag affected routes for review', 'Notify ethics team', 'Enable fairness monitoring alerts'],
            'resolution_steps': ['Audit training data for representation gaps', 'Retrain with balanced dataset', 'Implement fairness constraints'],
            'post_incident': ['Add fairness metrics to deployment gates', 'Schedule regular bias audits', 'Establish bias monitoring dashboard']
        },
        'error_rate': {
            'containment_actions': ['Activate error handling fallbacks', 'Increase logging verbosity', 'Alert on-call engineer'],
            'resolution_steps': ['Debug error patterns', 'Fix root cause in code/config', 'Deploy hotfix with rollback plan'],
            'post_incident': ['Add error scenario tests', 'Improve error handling coverage', 'Review deployment procedures']
        },
        'uptime_pct': {
            'containment_actions': ['Initiate failover to backup systems', 'Activate incident commander', 'Notify all stakeholders immediately'],
            'resolution_steps': ['Restore from backup', 'Rollback problematic deployment', 'Fix infrastructure issue'],
            'post_incident': ['Implement chaos engineering', 'Improve change management', 'Add redundancy layers']
        },
        'content_safety_score': {
            'containment_actions': ['Disable chatbot/content generation immediately', 'Escalate to content safety team', 'Preserve all generated content logs'],
            'resolution_steps': ['Audit training data for harmful examples', 'Review and fix safety guardrails', 'Implement content filtering layer'],
            'post_incident': ['Add content safety testing to CI/CD', 'Establish content moderation dashboard', 'Schedule regular safety audits']
        }
    }
    
    base = responses.get(metric, responses['error_rate'])
    result = {
        'containment_actions': list(base['containment_actions']),
        'resolution_steps': list(base['resolution_steps']),
        'post_incident': list(base['post_incident'])
    }
    
    # Severity adjustments
    if severity == 'S1':
        result['containment_actions'].insert(0, 'ACTIVATE WAR ROOM — all hands on deck')
        result['containment_actions'].append('Prepare regulatory notification')
    elif severity == 'S2':
        result['containment_actions'].insert(0, 'Alert management team and begin incident tracking')
    elif severity == 'S4':
        result['containment_actions'] = ['Monitor trend; escalate if worsens'] + result['containment_actions'][:1]
        result['resolution_steps'] = result['resolution_steps'][:1]  # Simplified for low severity
    
    # Compound incident logic: when multiple metrics breach simultaneously
    if 'COMPOUND' in metric:
        result['containment_actions'].insert(0, 'SYSTEM DEGRADATION DETECTED: Multiple metrics breached simultaneously')
        result['containment_actions'].append('Initiate full health check across all subsystems')
        if 'model_accuracy' in metric and 'error_rate' in metric:
            result['containment_actions'].insert(1, 'Activate full system failover')
            result['resolution_steps'].insert(0, 'Investigate for common root cause (load, corruption, misconfiguration)')
    
    return result

# Generate recommendations for unique incident types
if len(incident_df) > 0:
    unique_incidents = incident_df.drop_duplicates(subset=['metric', 'severity'])
    print("Response Recommendations:")
    print("=" * 70)
    for _, row in unique_incidents.iterrows():
        rec = recommend_response(row['metric'], row['severity'])
        print(f"\n{row['metric']} ({row['severity']}):")
        print(f"  Contain: {rec['containment_actions']}")
        print(f"  Resolve: {rec['resolution_steps']}")
        print(f"  Prevent: {rec['post_incident']}")

## Visualization (Pre-provided)

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 14))
fig.suptitle('TransitAI System Monitoring — Incident Detection', fontsize=14, fontweight='bold')

metrics = ['model_accuracy', 'response_latency_ms', 'bias_score', 'error_rate', 'uptime_pct', 'content_safety_score']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#e377c2']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax = axes[i]
    ax.plot(range(len(monitoring_data)), monitoring_data[metric], color=color, linewidth=1, alpha=0.8)
    
    t = thresholds[metric]
    ax.axhline(y=t['warning'], color='orange', linestyle='--', alpha=0.7, label=f"Warning ({t['warning']})")
    ax.axhline(y=t['critical'], color='red', linestyle='--', alpha=0.7, label=f"Critical ({t['critical']})")
    
    # Mark detected incidents
    if len(incident_df) > 0:
        metric_incidents = incident_df[incident_df['metric'] == metric]
        for _, inc in metric_incidents.iterrows():
            idx = monitoring_data[monitoring_data['timestamp'] == inc['timestamp']].index
            if len(idx) > 0:
                marker = 'v' if inc['severity'] == 'S1' else '^' if inc['severity'] == 'S2' else 's' if inc['severity'] == 'S3' else 'D'
                inc_color = 'red' if inc['severity'] == 'S1' else 'orange' if inc['severity'] == 'S2' else 'yellow' if inc['severity'] == 'S3' else 'lightblue'
                ax.plot(idx[0], inc['value'], marker, color=inc_color, markersize=8)
    
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=9)
    ax.legend(loc='upper right', fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transitai_monitoring.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved as transitai_monitoring.png")

## Summary

You implemented three key components of an incident detection system:
1. **Detection logic** — threshold-based monitoring with warning/critical levels
2. **Escalation decisions** — routing incidents to the right people based on severity
3. **Response recommendations** — containment, resolution, and prevention actions

In a production transit system, these would integrate with real-time monitoring tools (Prometheus, DataDog) and incident management platforms (PagerDuty, Jira).